<a href="https://colab.research.google.com/github/minji0620/sk_broadband/blob/main/Sales%20Talk/SalesTalk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
!pip install openai

In [ ]:
from openai import OpenAI
from getpass import getpass

In [ ]:
api_key = getpass('OpenAI API 키를 입력하세요: ')
client = OpenAI(api_key=api_key)

# completion = client.chat.completions.create(
#   model="gpt-4o-mini",
#   store=True,
#   messages=[
#     {"role": "user", "content": "write a haiku about ai"}
#   ]
# )

# print(completion.choices[0].message.content);


OpenAI API 키를 입력하세요: ··········


In [ ]:
df=pd.read_csv("/content/customer_feature_list.csv",encoding="utf-8-sig")

In [ ]:
# 각 특징에 대한 추천 이유 매핑
feature_to_reason = {
    "와이파이 접속 기기 대수가 많은 고객": "인터넷 이용량이 많아 다양한 콘텐츠 소비에 관심이 높을 수 있어 IPTV도 활용도가 높습니다.",
    "데이터 수신량이 많은 고객": "인터넷을 통한 콘텐츠 수신량이 많아 IPTV 서비스와의 시너지가 기대됩니다.",
    "데이터 송신량이 많은 고객": "활발한 데이터 사용은 미디어 소비 관심과 연결될 수 있어 IPTV 추가 이용에 적합합니다.",
    "장기고객": "오랜 기간 이용해주신 고객님께 IPTV 추가 가입 시 특별 혜택을 제공할 수 있습니다.",
    "윙즈(와이파이 확장기) 사용 고객": "인터넷 부가서비스를 이용하고 계시니, IPTV 추가 시 더 많은 고화질 콘텐츠를 안정적으로 즐기실 수 있습니다.",
    "sk 브로드밴드 와이파이 공유기 사용 고객": "SK브로드밴드 인터넷과 기기를 함께 사용 중이라 IPTV 추가 시 연동성과 품질이 우수합니다.",
    "재약정 이력이 있는 고객": "기존 계약을 연장해주신 만큼 IPTV 가입 시 추가 할인 혜택을 받으실 수 있습니다.",
    "기가인터넷 사용 고객": "고속 인터넷 환경을 보유하고 있어 IPTV 콘텐츠를 고화질로 끊김 없이 즐기실 수 있습니다.",
    "약정기간이 얼마 남지 않은 고객": "약정 만료 시점에 IPTV를 추가 가입하면 재약정과 함께 더 나은 조건을 적용받을 수 있습니다."
}

In [ ]:
# sales talk 생성용 프롬프트 만드는 함수
def build_prompt(feature_list):
    feature_items = [feature.strip() for feature in feature_list.split(",")]

    features_section = "\n".join([f"- {feature}" for feature in feature_items])
    reasons_section = "\n".join([
        f"- {feature}: {feature_to_reason.get(feature)}"
        for feature in feature_items
    ])

    prompt = f"""
현재 추천하려는 고객의 특성은 다음과 같아
{features_section}

그리고 각 특성에 대한 추천 이유는 다음과 같아
{reasons_section}

이 정보를 참고해서,
고객이 자연스럽게 이해할 수 있도록 IPTV 가입을 권유하는 세일즈 멘트를 한 문단으로 작성해줘.
회사의 입장이 아니라 고객의 입장에서, 편리성과 혜택을 부드럽게 강조해서 설득해줘.
참고로 와이파이 접속 기기 대수에서 가족 언급은 피해줘
"""
    return prompt.strip()

In [ ]:
print(build_prompt("장기고객, 데이터 수신량이 많은 고객, 기가인터넷 사용 고객"))

현재 추천하려는 고객의 특성은 다음과 같아
- 장기고객
- 데이터 수신량이 많은 고객
- 기가인터넷 사용 고객

그리고 각 특성에 대한 추천 이유는 다음과 같아
- 장기고객: 오랜 기간 이용해주신 고객님께 IPTV 추가 가입 시 특별 혜택을 제공할 수 있습니다.
- 데이터 수신량이 많은 고객: 인터넷을 통한 콘텐츠 수신량이 많아 IPTV 서비스와의 시너지가 기대됩니다.
- 기가인터넷 사용 고객: 고속 인터넷 환경을 보유하고 있어 IPTV 콘텐츠를 고화질로 끊김 없이 즐기실 수 있습니다.

이 정보를 참고해서,
고객이 자연스럽게 이해할 수 있도록 IPTV 가입을 권유하는 세일즈 멘트를 한 문단으로 작성해줘.
회사의 입장이 아니라 고객의 입장에서, 편리성과 혜택을 부드럽게 강조해서 설득해줘.


In [ ]:
SYSTEM_PROMPT = """
너는 SK브로드밴드의 방문 수리 및 영업직원이야.
SK브로드밴드는 인터넷과 IPTV를 주로 판매하는 회사야.
인터넷만 사용하고 있는 고객에게 IPTV 가입을 권유하는 자연스러운 영업멘트를 생성해줘.
"""

In [ ]:
# sales_talk 컬럼 미리 생성 (빈 값으로)
df['sales_talk'] = ""

# 0~9행만 선택해서 처리
for idx in range(10,20):  # 0부터 9까지
    row = df.iloc[idx]
    feature_list = row['feature_list']

    # 프롬프트 생성
    prompt = build_prompt(feature_list)

    response = client.chat.completions.create(
        model="gpt-4o",
        messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=300,
        seed=42
    )

    sales_talk= response.choices[0].message.content.strip()
    print(f"{idx+1}번 고객: {feature_list}")
    print(sales_talk)
    # 해당 인덱스에 결과 저장
    df.at[idx, 'sales_talk'] = sales_talk

11번 고객: 장기고객, 데이터 수신량이 많은 고객, 기가인터넷 이용 고객
안녕하세요 고객님, 오랜 기간 저희 서비스를 이용해주셔서 진심으로 감사드립니다. 고객님처럼 데이터 수신량이 많은 분들께는 IPTV 서비스가 특히 유용할 수 있습니다. 기가인터넷을 통해 빠르고 안정적인 연결을 이미 경험하고 계시니, IPTV를 추가하시면 더욱 풍부한 콘텐츠를 끊김 없이 즐기실 수 있습니다. 특별히 장기 고객님께는 IPTV 가입 시 특별 혜택도 제공해드리고 있으니, 더 많은 즐거움을 누리실 수 있는 기회를 놓치지 마세요!
12번 고객: 윙즈(와이파이 확장기) 사용 고객, 데이터 수신량이 많은 고객, 기가인터넷 이용 고객
안녕하세요! 현재 기가인터넷과 윙즈를 사용하고 계신데, 이를 통해 인터넷을 더욱 효율적으로 활용하고 계신 것 같아요. 특히 데이터 수신량이 많으신 만큼, IPTV를 추가하시면 고화질의 다양한 콘텐츠를 끊김 없이 즐기실 수 있습니다. IPTV는 인터넷과의 시너지를 통해 더욱 풍부한 경험을 제공해드릴 수 있어요. 지금처럼 안정적인 인터넷 환경에서, IPTV로 더 많은 즐거움을 누려보시는 건 어떨까요?
13번 고객: 기가인터넷 이용 고객
안녕하세요! 기가인터넷을 사용 중이시라면, IPTV를 함께 이용하시면 더욱 풍부한 엔터테인먼트 경험을 즐기실 수 있습니다. IPTV는 다양한 채널과 최신 영화, 드라마를 언제든지 쉽게 시청할 수 있는 서비스로, 기가인터넷의 빠른 속도와 결합하여 끊김 없이 고화질로 콘텐츠를 즐기실 수 있습니다. 특히, 원하는 프로그램을 놓치지 않고 다시 볼 수 있는 기능도 제공되어, 바쁜 일상 속에서도 편리하게 여가 시간을 활용하실 수 있습니다. 지금 가입하시면 특별한 혜택도 제공되니, 더 많은 즐거움을 누려보세요!
14번 고객: 데이터 수신량이 많은 고객, 기가인터넷 이용 고객, SK브로드밴드 와이파이 공유기 사용 고객
안녕하세요! 고객님께서 이미 SK브로드밴드의 기가인터넷과 와이파이 공유기를 사용하고 계시니, IPTV 서비스를 함께 이용하시면 

In [ ]:
# sales_talk 컬럼 미리 생성 (빈 값으로)
df['sales_talk'] = ""

# 0~9행만 선택해서 처리
for idx in range(10):  # 0부터 9까지
    row = df.iloc[idx]
    feature_list = row['feature_list']

    # 프롬프트 생성
    prompt = build_prompt(feature_list)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}],
        temperature=0.15,
        max_tokens=300
    )

    sales_talk= response.choices[0].message.content.strip()
    print(f"{idx+1}번 고객: {feature_list}")
    print(sales_talk)
    # 해당 인덱스에 결과 저장
    df.at[idx, 'sales_talk'] = sales_talk

1번 고객: 윙즈(와이파이 확장기) 사용 고객, 기가인터넷 이용 고객, 와이파이 접속 기기 대수가 많은 고객
안녕하세요, 고객님! 현재 윙즈를 사용하시면서 기가인터넷을 이용하고 계시니, IPTV를 추가하시면 더욱 풍부한 고화질 콘텐츠를 안정적으로 즐기실 수 있습니다. 특히, 여러 대의 기기를 동시에 사용하시는 고객님께는 IPTV가 더할 나위 없이 편리한 선택이 될 거예요. 다양한 채널과 영화, 드라마를 끊김 없이 감상하실 수 있어 가족 모두가 만족할 수 있는 즐거운 시청 경험을 제공해 드릴 수 있습니다. IPTV 가입을 고려해 보시는 건 어떠신가요?
2번 고객: 기가인터넷 이용 고객, 와이파이 접속 기기 대수가 많은 고객, SK브로드밴드 와이파이 공유기 사용 고객
안녕하세요, 고객님! 현재 기가인터넷을 이용하고 계시고, 여러 기기에서 와이파이를 사용하고 계신 만큼, IPTV를 추가하시면 더욱 편리하게 다양한 콘텐츠를 즐기실 수 있습니다. 특히, SK브로드밴드의 IPTV는 고화질의 생생한 영상과 다양한 채널을 제공하여 가족 모두가 만족할 수 있는 선택이 될 거예요. 또한, 와이파이 공유기를 사용하고 계시니, 안정적인 인터넷 속도로 끊김 없이 영화나 드라마를 감상하실 수 있습니다. IPTV와 함께라면, 집에서의 여가 시간이 더욱 풍성해질 거라 확신합니다!
3번 고객: 데이터 송신량이 많은 고객, 기가인터넷 이용 고객, SK브로드밴드 와이파이 공유기 사용 고객
안녕하세요! 고객님, 현재 기가인터넷을 이용하시고 데이터 송신량이 많으시다고 하셨는데, 이런 활발한 인터넷 사용은 다양한 미디어 콘텐츠를 즐기고 싶다는 뜻일 수 있습니다. 그래서 IPTV를 추가하시면, 고화질의 다양한 영화와 드라마, 스포츠 중계 등을 손쉽게 즐기실 수 있어요. 특히 SK브로드밴드의 와이파이 공유기를 사용하고 계시니, 안정적인 연결로 끊김 없이 시청할 수 있는 장점도 있습니다. IPTV 가입을 통해 더 풍성한 미디어 경험을 누려보시는 건 어떠신가요?
4번 고객: 와이파이 접속 기기 대수가 많은

In [ ]:
prompt='''
    고객별 특성과 특성에 따른 추천 근거야
    와이파이 접속 기기 대수가 많은 고객: 인터넷 이용량이 많아 다양한 콘텐츠 소비에 관심이 높을 수 있어 IPTV도 활용도가 높습니다.,
    데이터 수신량이 많은 고객: 인터넷을 통한 콘텐츠 수신량이 많아 IPTV 서비스와의 시너지가 기대됩니다.,
    데이터 송신량이 많은 고객: 활발한 데이터 사용은 미디어 소비 관심과 연결될 수 있어 IPTV 추가 이용에 적합합니다.,
    장기고객: 오랜 기간 이용해주신 고객님께 IPTV 추가 가입 시 특별 혜택을 제공할 수 있습니다.,
    윙즈(와이파이 확장기) 사용 고객: 인터넷 부가서비스를 이용하고 계시니, IPTV 추가 시 더 많은 고화질 콘텐츠를 안정적으로 즐기실 수 있습니다.,
    sk 브로드밴드 와이파이 공유기 사용 고객: SK브로드밴드 인터넷과 기기를 함께 사용 중이라 IPTV 추가 시 연동성과 품질이 우수합니다.,
    재약정 이력이 있는 고객: 기존 계약을 연장해주신 만큼 IPTV 가입 시 추가 할인 혜택을 받으실 수 있습니다.,
    기가인터넷 사용 고객: 고속 인터넷 환경을 보유하고 있어 IPTV 콘텐츠를 고화질로 끊김 없이 즐기실 수 있습니다.,
    약정기간이 얼마 남지 않은 고객: 약정 만료 시점에 IPTV를 추가 가입하면 재약정과 함께 더 나은 조건을 적용받을 수 있습니다.
    위 고객별 특성과 추천 근거를 참고해 다음과 같은 특성의 고객에게 고객이 자연스럽게 이해할 수 있도록 IPTV 가입을 권유하는 세일즈 멘트를 한 문단으로 작성해줘.\n
    '''
while True:
    user_input = input("당신: ")

    if user_input.lower() in ["종료", "exit", "quit"]:
        print("대화를 종료합니다.")
        break

    # 2. GPT한테 messages 전체를 보내고
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt+user_input}],
        temperature=0.2,
        max_tokens=500
    )

    assistant_reply = response.choices[0].message.content
    print(f"GPT: {assistant_reply}")

당신: 약정기간이 얼마 남지 않으셨고, sk 와이파이 공유기를 사용하시며, 장기고객이야
GPT: 안녕하세요, 고객님! 오랜 기간 저희 SK브로드밴드를 이용해 주셔서 정말 감사합니다. 현재 약정기간이 얼마 남지 않으신데, 이 기회에 IPTV를 추가 가입하시면 재약정과 함께 더 나은 조건으로 혜택을 받으실 수 있습니다. 또한, SK 와이파이 공유기를 사용하고 계시기 때문에 IPTV와의 연동성이 뛰어나고, 고화질 콘텐츠를 안정적으로 즐기실 수 있습니다. 다양한 프로그램과 영화, 스포츠 등 풍성한 콘텐츠를 통해 더욱 풍요로운 미디어 경험을 누려보시는 건 어떠세요? 고객님의 소중한 의견을 듣고 싶습니다!
당신: 데이터 수신량이 높으시고, 기가 인터넷을 사용하고 계시며, 재약정 이력이 있으신 고객에게 sales talk 추천해줘
GPT: 안녕하세요, 고객님! 현재 데이터 수신량이 높으시고 기가 인터넷을 사용하고 계신 만큼, 고화질 콘텐츠를 끊김 없이 즐기실 수 있는 최적의 환경을 갖추고 계십니다. 또한, 재약정 이력이 있으신 고객님께는 IPTV 가입 시 특별 할인 혜택을 제공해드릴 수 있습니다. 다양한 영화와 드라마, 스포츠 중계 등 풍부한 콘텐츠를 IPTV로 즐기시면, 더욱 풍성한 미디어 경험을 누리실 수 있을 거예요. 고객님께서 관심 있으시다면, IPTV 서비스에 대해 더 자세히 안내해드리겠습니다!
당신: 와이파이 접속 기기 대수가 많고, 윙즈 사용 고객에게 영업 멘트 만들어줘
GPT: 안녕하세요, 고객님! 현재 여러 기기를 통해 와이파이에 접속하고 계신데요, 이렇게 다양한 기기를 사용하신다면 콘텐츠 소비에 대한 관심도 높으실 것 같습니다. 특히, 저희 SK브로드밴드의 IPTV 서비스를 추가하시면 고화질의 다양한 콘텐츠를 안정적으로 즐기실 수 있습니다. 또한, 고객님께서 사용 중이신 윙즈와 함께하면 더욱 원활한 스트리밍 환경을 제공받으실 수 있어요. IPTV 가입 시 특별 혜택도 준비되어 있으니, 한번 고려해보시는 건 어떨까요?
당신: 데이터 송신량이 많으시고, sk

In [ ]:
import re

prompt_info = '''
고객별 특성과 특성에 따른 추천 근거야
와이파이 접속 기기 대수가 많은 고객: 인터넷 이용량이 많아 다양한 콘텐츠 소비에 관심이 높을 수 있어 IPTV도 활용도가 높습니다.,
데이터 수신량이 많은 고객: 인터넷을 통한 콘텐츠 수신량이 많아 IPTV 서비스와의 시너지가 기대됩니다.,
데이터 송신량이 많은 고객: 활발한 데이터 사용은 미디어 소비 관심과 연결될 수 있어 IPTV 추가 이용에 적합합니다.,
장기고객: 오랜 기간 이용해주신 고객님께 IPTV 추가 가입 시 특별 혜택을 제공할 수 있습니다.,
윙즈(와이파이 확장기) 사용 고객: 인터넷 부가서비스를 이용하고 계시니, IPTV 추가 시 더 많은 고화질 콘텐츠를 안정적으로 즐기실 수 있습니다.,
sk 브로드밴드 와이파이 공유기 사용 고객: SK브로드밴드 인터넷과 기기를 함께 사용 중이라 IPTV 추가 시 연동성과 품질이 우수합니다.,
재약정 이력이 있는 고객: 기존 계약을 연장해주신 만큼 IPTV 가입 시 추가 할인 혜택을 받으실 수 있습니다.,
기가인터넷 사용 고객: 고속 인터넷 환경을 보유하고 있어 IPTV 콘텐츠를 고화질로 끊김 없이 즐기실 수 있습니다.,
약정기간이 얼마 남지 않은 고객: 약정 만료 시점에 IPTV를 추가 가입하면 재약정과 함께 더 나은 조건을 적용받을 수 있습니다.

위 고객별 특성과 추천 근거를 참고해 다음과 같은 특성의 고객에게 고객이 자연스럽게 이해할 수 있도록 IPTV 가입을 권유하는 세일즈 멘트를 한 문단으로 작성해줘.
'''

In [ ]:
while True:
    user_input = input("당신: ")

    if user_input.lower() in ["종료", "끝", "quit"]:
        print("대화를 종료합니다.")
        break

    # 사용자 입력에서 숫자(고객 번호) 추출
    numbers = re.findall(r'\d+', user_input)

    if numbers:
        for num in numbers:
            customer_index = int(num)

            # customer_index가 df에 있는지 확인
            if customer_index in df['customer_index'].values:
                feature_list = df[df['customer_index'] == customer_index]['feature_list'].values[0]
                user_prompt = feature_list

                # GPT 호출
                response = client.chat.completions.create(
                    model="gpt-4o",
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": prompt_info + user_prompt}
                    ],
                    temperature=0.2,
                    max_tokens=500,seed=42
                )

                assistant_reply = response.choices[0].message.content
                print(f"\n [고객 {customer_index}번 세일즈 토크]")
                print(assistant_reply)
                print("\n" + "-"*50 + "\n")

            else:
                print(f"⚠️ 고객 {customer_index}번은 데이터에 없습니다.")
    else:
        user_prompt = user_input  # 그냥 사용자가 입력한 대로 사용

        # GPT 호출
        response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": SYSTEM_PROMPT},{"role": "user", "content": prompt_info + user_prompt}]
        ,temperature=0.2,max_tokens=500,seed=42)

        assistant_reply = response.choices[0].message.content
        print(f"GPT: {assistant_reply}")


당신: 11번

 [고객 11번 세일즈 토크]

안녕하세요 고객님, SK브로드밴드를 이용해주셔서 감사합니다. 고객님께서는 현재 기가인터넷을 사용 중이시고, 데이터 수신량이 많으신 것으로 확인되었습니다. 또한, SK브로드밴드의 와이파이 공유기를 사용하고 계셔서 저희 서비스와의 연동성이 뛰어납니다. 이러한 환경에서는 IPTV를 추가로 이용하시면 고화질의 다양한 콘텐츠를 끊김 없이 즐기실 수 있습니다. IPTV는 고객님의 인터넷 사용 패턴에 맞춰 더욱 풍부한 미디어 경험을 제공할 수 있으니, 한번 고려해보시는 건 어떨까요? 추가 가입 시 특별한 혜택도 준비되어 있으니, 언제든지 문의해 주세요. 감사합니다.

--------------------------------------------------

당신: 11번

 [고객 11번 세일즈 토크]

안녕하세요 고객님, SK브로드밴드를 이용해주셔서 감사합니다. 고객님께서는 기가인터넷을 사용 중이시고, 데이터 수신량이 많으신 것으로 확인됩니다. 이러한 환경에서는 다양한 콘텐츠를 고화질로 끊김 없이 즐기실 수 있는 IPTV 서비스가 큰 만족을 드릴 수 있을 것 같습니다. 특히, SK브로드밴드 와이파이 공유기를 사용하고 계시니, IPTV와의 연동성이 뛰어나 더욱 안정적인 시청 환경을 제공해드립니다. IPTV를 통해 더욱 풍부한 콘텐츠를 경험해보시는 건 어떠실까요? 추가 가입 시 특별 혜택도 준비되어 있으니, 언제든지 문의해 주세요. 감사합니다.

--------------------------------------------------

당신: 11번

 [고객 11번 세일즈 토크]

안녕하세요 고객님, SK브로드밴드를 이용해주셔서 감사합니다. 고객님께서는 기가인터넷을 사용 중이시고, 데이터 수신량이 많으신 것으로 확인됩니다. 이러한 환경에서는 다양한 콘텐츠를 고화질로 끊김 없이 즐기실 수 있는 IPTV 서비스가 큰 도움이 될 수 있습니다. 또한, SK브로드밴드 와이파이 공유기를 사용하고 계시니 IPTV와의 연동성이 뛰어나 

In [ ]:
while True:
    user_input = input("당신: ")

    if user_input.lower() in ["종료", "끝", "quit"]:
        print("대화를 종료합니다.")
        break

    numbers = re.findall(r'\d+', user_input)

    if numbers:
        for num in numbers:
            customer_index = int(num)

            if customer_index in df['customer_index'].values:
                feature_list = df[df['customer_index'] == customer_index]['feature_list'].values[0]
                user_prompt = feature_list

                print(f"\n [고객 {customer_index}번 세일즈 토크]")

                response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": prompt_info + user_prompt}
                    ],
                    temperature=0.2,
                    max_tokens=500,
                    stream=True  # ✨ 추가
                )

                # 스트리밍 응답 읽기
                for chunk in response:
                  content = chunk.choices[0].delta.content
                  if content:
                    print(content, end="", flush=True)

                print("\n" + "-"*50 + "\n")

            else:
                print(f"⚠️ 고객 {customer_index}번은 데이터에 없습니다.")

    else:
        user_prompt = user_input

        print("\nGPT:")

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt_info + user_prompt}
            ],
            temperature=0.2,
            max_tokens=500,
            stream=True  # ✨ 추가
        )

        # 스트리밍 응답 읽기
        for chunk in response:
          content = chunk.choices[0].delta.content
          if content:
            print(content, end="", flush=True)

        print("\n" + "-"*50 + "\n")


당신: 장기고객이시고 데이터 송신량이 많은 고객의 적절한 영업멘트 생성해줘

GPT:
안녕하세요, 고객님! 오랜 기간 저희 SK브로드밴드를 이용해 주셔서 정말 감사드립니다. 고객님께서는 데이터 송신량이 많으신데, 이는 미디어 소비에 대한 관심이 높으신 것으로 보입니다. 그래서 IPTV 서비스를 추천드리고 싶습니다. IPTV를 추가하시면 다양한 고화질 콘텐츠를 안정적으로 즐기실 수 있고, 장기 고객님께는 특별한 혜택도 제공해드릴 수 있습니다. 고객님의 인터넷 환경과 잘 어우러져 더욱 풍성한 콘텐츠 경험을 누리실 수 있을 거예요. 관심 있으시면 자세히 안내해드리겠습니다!
--------------------------------------------------

당신: 종료
대화를 종료합니다.
